---
title: Data Access
execute:
  enabled: true
---

This page's code cells are **executed live** by Quarto (via Jupyter) every time the
docs are built. `q.data` collects everything data-related under one namespace:

| Submodule | What it is |
|---|---|
| `q.data.datasets` | prepackaged sample OHLCV data (AAPL, SPY, BTC-USD), works offline |
| `q.data.local` (`q.data.load`/`q.data.save`) | load/save local parquet or csv files |
| `q.data.sources` | downloadable market data vendors and generic backends, one submodule each |

Live network calls (`q.data.sources.yfinance`, `.binance`) aren't executed on this
page so the docs build has no network dependency — see the code blocks below for
how to call them yourself.

## Prepackaged sample datasets

`q.data.datasets.load` ships daily OHLCV parquet files for a handful of symbols so
tests, demos, and tutorials (like this one, and the [Feature Engineering](feat.ipynb)
and [Return Statistics](stats.ipynb) pages) don't need network access:

In [1]:
import qrt as q

q.data.datasets.AVAILABLE

('aapl', 'btcusd', 'spy')

In [2]:
aapl = q.data.datasets.load("aapl")
aapl.tail()

,open,high,low,close,volume
datetime,,,,,
2026-07-13,317.019989,323.450012,315.779999,317.309998,43257800
2026-07-14,313.760010,316.190002,311.910004,314.859985,36336800
2026-07-15,317.619995,328.730011,317.320007,327.500000,60957600
2026-07-16,328.010010,334.679993,326.790009,333.260010,62970600
2026-07-17,331.980011,334.980011,329.000610,333.739990,63325386


These bundled files are refreshed via `make datasets` before a release, so they're
reasonably current but not guaranteed to have today's bar — fetch live data through
`q.data.sources` if you need that.

## Local files

`q.data.load`/`q.data.save` dispatch on the file suffix: `.parquet` (via DuckDB) or
`.csv` (via pandas). A quick round trip through a temp directory:

In [3]:
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "aapl.parquet"
    q.data.save(aapl.tail(30), path, index=True)
    roundtrip = q.data.load(path, index="datetime")

roundtrip.tail()

,open,high,low,close,volume
datetime,,,,,
2026-07-13,317.019989,323.450012,315.779999,317.309998,43257800
2026-07-14,313.760010,316.190002,311.910004,314.859985,36336800
2026-07-15,317.619995,328.730011,317.320007,327.500000,60957600
2026-07-16,328.010010,334.679993,326.790009,333.260010,62970600
2026-07-17,331.980011,334.980011,329.000610,333.739990,63325386


## Market data sources

Each vendor/backend is its own submodule under `q.data.sources`, all returning the
same lowercase-column OHLCV `DataFrame` layout consumed by `q.feat` and `q.stats`.
These examples hit the network, so they're shown but not executed here:

```python
# Yahoo Finance — cached locally as parquet after the first call
ohlc = q.data.sources.yfinance.read("AAPL", "2024-01-01", "2025-01-01", "1d")

# Binance futures — daily trade dumps aggregated into OHLC bars, also cached
ohlc = q.data.sources.binance.read("BTCUSDT", "2025-01-01", "2025-01-07", "1h")
```

`q.data.sources.duckdb` is different: rather than a fixed schema/vendor, it reads
and writes arbitrary tables in a DuckDB database, keyed by `symbol` and `datetime`.
It works offline (`:memory:` by default), so we can run it live:

In [4]:
table = aapl.tail(30).reset_index()
table.insert(1, "symbol", "AAPL")

db = q.data.sources.duckdb.connect()  # in-memory
db.write(table)
db.read("AAPL", table["datetime"].min(), table["datetime"].max()).tail()

,datetime,symbol,open,high,low,close,volume
25,2026-07-13,AAPL,317.019989,323.450012,315.779999,317.309998,43257800
26,2026-07-14,AAPL,313.760010,316.190002,311.910004,314.859985,36336800
27,2026-07-15,AAPL,317.619995,328.730011,317.320007,327.500000,60957600
28,2026-07-16,AAPL,328.010010,334.679993,326.790009,333.260010,62970600
29,2026-07-17,AAPL,331.980011,334.980011,329.000610,333.739990,63325386
